In [0]:
%pip install azure-eventhub

### Step 1: Event Hub Producer (Message Ingestion Engine)
* Authenticates to Azure Key Vault scope to retrieve the Event Hub Connection String securely.
* Generates synthetic real-time event payloads and sends them in batches to `natalkamartinuk55-stream-hub`.

In [0]:
import json
import uuid
from datetime import datetime
from azure.eventhub import EventHubProducerClient, EventData

secret_scope = "natalkamartinuk55_scope"
secret_key = "natalkamartinuk55-eventhub-conn"

conn_str = dbutils.secrets.get(scope=secret_scope, key=secret_key)
eventhub_name = "natalkamartinuk55-stream-hub"

producer = EventHubProducerClient.from_connection_string(
    conn_str=conn_str, 
    eventhub_name=eventhub_name
)

events_data = []
categories = ["Music", "Gaming", "News", "Education", "Entertainment"]

for i in range(25):
    event_payload = {
        "event_id": str(uuid.uuid4()),
        "video_id": f"VID_{1000 + i}",
        "user_id": f"USER_{500 + (i % 10)}",
        "category": categories[i % len(categories)],
        "watch_duration_sec": 45 + (i * 7),
        "event_timestamp": datetime.utcnow().isoformat()
    }
    events_data.append(event_payload)

event_batch = producer.create_batch()
for event in events_data:
    event_batch.add(EventData(json.dumps(event)))

with producer:
    producer.send_batch(event_batch)



### Step 2: Spark Structured Streaming Consumer (Event Ingestion Engine)
* Reads real-time message streams from Azure Event Hubs using native Kafka-compatible protocol over SASL/SSL.
* Deserializes JSON payloads into structured PySpark schemas.
* Appends technical audit metadata (`_eventhub_offset`, `_eventhub_partition`, `_ingestion_timestamp`).
* Persists stream micro-batches into `natalkamartinuk55_bronze.eventhub_youtube_bronze` using `availableNow=True` for cost-effective execution.

In [0]:
from pyspark.sql.functions import col, from_json, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

secret_scope = "natalkamartinuk55_scope"
secret_key = "natalkamartinuk55-eventhub-conn"
eventhub_name = "natalkamartinuk55-stream-hub"

conn_str = dbutils.secrets.get(scope=secret_scope, key=secret_key)
eh_namespace = "evhua5816bd"
eh_bootstrap_servers = f"{eh_namespace}.servicebus.windows.net:9093"

eh_jaas_config = (
    f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
    f'username="$ConnectionString" '
    f'password="{conn_str}";'
)

event_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("video_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("category", StringType(), True),
    StructField("watch_duration_sec", IntegerType(), True),
    StructField("event_timestamp", StringType(), True)
])

df_event_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", eh_bootstrap_servers)
    .option("subscribe", eventhub_name)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config", eh_jaas_config)
    .option("kafka.request.timeout.ms", "60000")
    .option("kafka.session.timeout.ms", "30000")
    .option("kafka.group.id", "$Default")
    .option("startingOffsets", "earliest")
    .option("failOnDataLoss", "false")
    .load()
)

df_bronze_stream = (
    df_event_raw
    .select(
        from_json(col("value").cast("string"), event_schema).alias("data"),
        col("offset").alias("_eventhub_offset"),
        col("partition").alias("_eventhub_partition"),
        col("timestamp").alias("_eventhub_enqueued_timestamp")
    )
    .select("data.*", "_eventhub_offset", "_eventhub_partition", "_eventhub_enqueued_timestamp")
    .withColumn("_ingestion_timestamp", current_timestamp())
)

target_bronze_table = "dbr_dev_ua5816bd.natalkamartinuk55_bronze.eventhub_youtube_bronze"
eh_checkpoint_path = "abfss://natalkamartinuk55@dlsua5816bd.dfs.core.windows.net/eventhub_source/stream_checkpoint_group_fix"

query_eh = (
    df_bronze_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", eh_checkpoint_path)
    .trigger(availableNow=True)
    .toTable(target_bronze_table)
)

query_eh.awaitTermination()

In [0]:

df_check = spark.table("dbr_dev_ua5816bd.natalkamartinuk55_bronze.eventhub_youtube_bronze")
display(df_check.limit(10))

---
## Architectural Insights & Operational Semantics

### 1. UDF Justification Note
* Transformations in this pipeline rely strictly on PySpark native built-in Catalyst expressions (`from_json`, `cast`, `current_timestamp`).
* Custom Python UDFs were intentionally omitted to avoid JVM-to-Python process serialization overhead and retain Whole-Stage Code Generation performance during high-throughput streaming.

### 2. Processing Semantics: Exactly-Once vs At-Least-Once
* **At-Least-Once:** Streaming sources guarantee that records are not lost during node crashes, but network retries can lead to downstream duplicates.
* **End-to-End Exactly-Once:** Achieved in this pipeline through the tri-part architecture:
  1. Offset-replayable streaming source (Azure Event Hubs).
  2. Deterministic offset commits tracked in persistent WAL (`checkpointLocation`).
  3. Transactional, ACID-compliant idempotent sink (Delta Lake ACID log commits).

### 3. Fault Tolerance & Checkpointing
* The `checkpointLocation` continuously commits processed partition offsets and schema state to ADLS Gen2 storage.
* Upon compute failure or cluster restart, the engine reads transaction state and resumes strictly from the last uncommitted offset, preventing skipped or duplicated records.